# Lekcia 18: Zaistenie AI agentov pomocou kryptografických potvrdení

## Praktický zápisník

Tento zápisník prechádza štyrmi cvičeniami:

1. **Podpíš svoj prvý doklad** pre volanie nástroja agenta a over ho.
2. **Manipuluj s dokladom** a sleduj, ako zlyhá overenie.
3. **Vytvor reťazec troch dokladov** a potvrď integritu reťazca.
4. **Zabali volanie nástroja Microsoft Agent Framework** tak, aby každá akcia vydávala doklad.

Všetky kryptografické primitívy sú importované zo starostlivo udržiavaných knižníc (`pynacl` pre Ed25519, `jcs` pre RFC 8785 kanonický JSON, `hashlib` zo štandardnej knižnice Pythonu pre SHA-256). Logika dokladu sama o sebe je obyčajný Python, ktorý môžete čítať a upravovať.

Spúšťajte bunky v poradí. Každá sekcia je krátka a samostatná.


## Nastavenie

Nainštalujte dve závislosti. Obe majú povolujúce licencie (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Pomocné nástroje

Tieto dva pomocníci sa zaoberajú base64url kódovaním (bez doplnenia) a SHA-256 hashovaním ľubovoľných objektov. Zvyšok poznámkového bloku tak zostáva zameraný na samotnú logiku príjmu.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Sekcia 1: Podpíšte svoj prvý doklad

Predstavme si, že náš agent pre **Contoso Travel** práve vyhľadal lety z Sydney do Los Angeles pre zákazníka. Chceme zaznamenať tento hovor s nástrojom ako podpísaný doklad, aby ho budúci audítor mohol overiť bez toho, aby nám musel dôverovať.

### Krok 1.1: Vygenerujte podpisovací kľúč

V produkcii by podpisovací kľúč agenta žil v hardvérovom bezpečnostnom module (HSM), Azure Key Vault alebo podobnom chránenom úložisku. Pre túto lekciu vygenerujeme nový kľúč v pamäti.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): g3SyD_ecOKa1L8RQ79-pDy9em81H-O_jzp9VG4a3EP0


### Krok 1.2: Vytvorte náklad pre potvrdenie

Náklad obsahuje všetko, čo chceme, aby potvrdenie potvrdilo: kto konal, aký nástroj, s akými argumentmi, čo sa vrátilo, podľa akej politiky a kedy. Argumenty a výsledok zahashujeme, namiesto toho, aby sme ich zahrnuli priamo, aby potvrdenie neunikalo citlivý obsah.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-08-18T04:55:21Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Krok 1.3: Podpíšte a zostavte potvrdenie

Tri kroky:

1. Kanonizujte náklad pomocou JCS tak, aby dve implementácie vytvárajúce rovnaké logické potvrdenie produkovali bajtovo identické dáta.
2. Podpíšte kanonické bajty priamo súkromným kľúčom Ed25519. PureEdDSA interné hašuje správu, takže ďalšie predhašovanie by zmenilo protokol.

Podpis sa potom pripojí k pôvodnému nákladu, aby vzniklo finálne potvrdenie.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-08-18T04:55:21Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "Eu3MmrOSdGq2DuemkjNSPKfz5xbr18okNeTU11NJu2usnsnKtC-pjq2PZM1Oap-WXdVgYkL4iW6SeWPSZ2M9BQ",
    "public_key": "g3SyD_ecOKa1L8RQ79-pDy9em81H-O_jzp9VG4a3EP0"
  }
}


### Krok 1.4: Overenie potvrdenia

Overenie zvracia proces. Odstránime podpis, znovu vypočítame kanonické bajty a skontrolujeme podpis voči verejnému kľúču v potvrdení.

Auditor vykonávajúci toto overenie nepotrebuje od nás nič okrem samotného potvrdenia. Žiadnu službu na zavolanie, žiadny adresár kľúčov na dotazovanie, žiadnu dôveru.


In [7]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(canonical, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt)}")

Receipt is valid: True
Pre-hashed receipt valid: False


Mali by ste vidieť `Receipt is valid: True` a `Pre-hashed receipt valid: False`. Pozitívny prípad dokazuje, že cesta priameho podpisu JCS funguje; negatívna kontrola robí pravidlo rozsahu podpisu vykonateľným namiesto toho, aby zostalo len ako opis.


## Sekcia 2: Manipulácia s účtenkou

Celý účel účteniek je ten, že sú odolné voči manipulácii. Poďme to dokázať.

Upravíme presne jeden znak na účtenke a budeme sledovať zlyhanie overenia.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Čo sa práve stalo?

Keď sme zmenili `policy_id`, zmenili sa kanonické bajty. Podpis (ktorý sa vytváral nad pôvodnými kanonickými bajtami) už nepasuje. Overenie správne vrátilo `False`.

Nie je možné upraviť žiadne pole potvrdenia a pritom ho stále potvrdiť, pokiaľ útočník nemá súkromný kľúč. Pokiaľ je súkromný kľúč uložený v trezore kľúčov a verejný kľúč je zverejnený, manipuláciu nie je možné ukryť.

Vyskúšajte to sami: zmeňte `tool_name` alebo `agent_id` alebo `timestamp` v bunku hore a spustite znova. Každá zmena vytvorí neplatné potvrdenie.


## Časť 3: Reťazenie potvrdení

Jedno potvrdenie chráni jednu akciu. Väčšina agentov vykoná mnoho akcií. Aby bola celá sekvencia chránená pred neoprávnenou manipuláciou, prepojíme každé potvrdenie s predchádzajúcim tak, že do novej platby potvrdzujúceho dokumentu zahrnieme hash predchádzajúceho potvrdenia.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Ak niekto odstráni alebo zmení poradie potvrdení, reťaz sa práve v tom bode preruší. Overenie akéhokoľvek neskoršieho potvrdenia zlyhá, pretože jeho `previous_receipt_hash` už nezodpovedá skutočnému hashu svojho predchodcu.


In [9]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [10]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:d54d1e138bb080dd6bd825d9f015efc74ad6c7f78ee9504d0b68b39d0733b39c
Receipt 2: tool=confirm_booking, prev=sha256:9fbe5dc2ffcffd7d167ef13a744aecd578e474fb536cb7104ddb1a359607f2f6


In [11]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Teraz prerušte reťazec manipuláciou so stredným potvrdením a znova ho overte. Pozmenené potvrdenie neprejde kontrolou podpisu, A ďalšie potvrdenie neprejde kontrolou reťazového odkazu (pretože jeho `previous_receipt_hash` už neodpovedá hashu pozmeneného stredného potvrdenia).


In [12]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Doklad 0 stále prechádza overením (nebol upravený a nemá predchodcu, od ktorého by závisel). Doklad 1 zlyháva pri kontrole podpisu, pretože sme zmenili `tool_args_hash`. Doklad 2 zlyháva pri kontrole reťazového odkazu, pretože jeho `previous_receipt_hash` bol vypočítaný na základe pôvodného (teraz upraveného) dokladu 1.

Aj keby útočník prepodpísal upravený doklad 1 (čo nemôže urobiť bez súkromného kľúča), nezrovnalosť reťazového odkazu v doklade 2 by stále odhalila manipuláciu. Na skrytie zmeny by útočník musel prepodpísať každý doklad od bodu úpravy ďalej, čo si vyžaduje vlastníctvo súkromného kľúča.


## Časť 4: Zabaľte volanie nástroja agenta do podpisu potvrdenia

Pri reálnom nasadení nechcete, aby si každý autor agenta pamätal zavolať `make_receipt`. Chcete, aby podpisovanie potvrdení bolo automatické pri každom vyvolaní nástroja.

Tu je najjednoduchší vzor: obalová trieda, ktorá prijíma akúkoľvek volateľnú funkciu nástroja a vracia jej verziu emitujúcu potvrdenie. Tento vzor sa prispôsobí akémukoľvek agentnému frameworku, vrátane Microsoft Agent Framework (`agent_framework.foundry`).

Ak nemáte nastavený projekt Microsoft Foundry, miestna ukážka nižšie stále demonštruje tento vzor.


In [13]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [14]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Integrácia s Microsoft Agent Framework

Wrapper `ReceiptedTool` vyššie je nezávislý na frameworku. Ak ho chcete použiť vo vnútri agenta vytvoreného pomocou Microsoft Agent Framework, zaregistrujte zabalenú funkciu ako nástroj. Náčrt (nahradili by ste falošný príklad reálnou registráciou nástroja Microsoft Foundry):

```python
# Pseudokód ukazujúci tvar integrácie.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="Ste agent spoločnosti Contoso Travel ...",
#     tools=[receipted_lookup],   # zabalený nástroj, nie surová funkcia
# )
# response = agent.run("Nájdi lety zo Sydney do Los Angeles v júni.")
#
# # Po spustení má každý nástroj volaný agentom podpísaný doklad:
# audit_chain = receipted_lookup.receipts
```

Agentný framework nemusí vedieť nič o potvrdeniach. Podpisovanie potvrdení je zabalené okolo nástroja, nie integrované priamo do frameworku. Takto pridáte pôvodnosť do existujúceho kódu agenta bez jeho prepísania.


## Rekapitulácia a rozširovacia výzva

Máte:

- Vygenerovanú Ed25519 párovú kľúč.
- Vytvorený a podpísaný doklad o volaní agenta.
- Overený doklad offline pomocou iba verejného kľúča.
- Zmenený doklad a pozorované zlyhanie overenia.
- Vytvorenú sekvenciu troch dokladov spojených hašom.
- Zmenenú stred reťazca a pozorované zlyhanie podpisu aj prepojenia reťazca.
- Zabalil funkciu nástroja s automatickým podpisovaním dokladu.

**Rozširovacia výzva.** Rozšírte schému dokladu o pole `request_id` (UUID pre distribučné trasovanie). Aktualizujte `make_receipt`, aby ho obsahovalo, a potvrďte, že doklady sa stále overujú end-to-end. Potom zmeňte hodnotu po podpise a overenie by malo zlyhať. Toto vás núti pochopiť, ako každý bajt kanonického kódovania prispieva k podpisu.

**Dôležitá hranica.** Doklady dokazujú tri veci a iba tri veci: pripísanie (tento kľúč podpísal tento obsah), integritu (obsah sa od podpisu nezmenil) a poradie (tento doklad prišiel po tom doklade). Nepotvrdzujú, že akcia agenta bola správna, že politika pomenovaná v `policy_id` bola skutočne vyhodnotená, ani že agent dodržal všetky pravidlá. Doklady sú základom. Riadenie je systém, ktorý na tom staviate.

Prečítajte si znova README lekcie s touto hranicou na pamäti. Najčastejšou chybou tímov pri dokladoch je predpoklad, že „máme doklady“ znamená „sme riadení“. Nie je to pravda. Doklady umožňujú audit správania agenta. Nerobia ho správnym.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
